In [1]:
#Importing Labraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import joblib

In [2]:
# STEP 2: Load Dataset
# Replace with your dataset path
# df = pd.read_csv('coupon_data.csv')

# Sample simulated dataset
np.random.seed(42)
df = pd.DataFrame({
    'age': np.random.randint(18, 60, 500),
    'income': np.random.randint(20000, 100000, 500),
    'gender': np.random.choice(['Male', 'Female'], 500),
    'coupon_type': np.random.choice(['Food', 'Travel', 'Electronics'], 500),
    'offer_duration': np.random.randint(1, 10, 500),
    'store_distance': np.random.randint(1, 20, 500),
    'past_coupon_use': np.random.randint(0, 15, 500),
    'coupon_used': np.random.choice([0,1], 500)
})


In [3]:
# STEP 3: Define Features and Target
X = df.drop('coupon_used', axis=1)
y = df['coupon_used']

In [4]:
# STEP 4: Identify Column Types
num_cols = ['age', 'income', 'offer_duration', 'store_distance', 'past_coupon_use']
cat_cols = ['gender', 'coupon_type']


In [5]:
# STEP 5: Preprocessing Pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])


In [6]:
# STEP 6: Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
# STEP 7: Logistic Regression Model
log_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:,1]

print('Logistic Regression Results')
print('Accuracy:', accuracy_score(y_test, y_pred))
print('ROC AUC:', roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Logistic Regression Results
Accuracy: 0.48
ROC AUC: 0.4951660361496427
              precision    recall  f1-score   support

           0       0.40      0.64      0.49        39
           1       0.62      0.38      0.47        61

    accuracy                           0.48       100
   macro avg       0.51      0.51      0.48       100
weighted avg       0.53      0.48      0.48       100



In [8]:
# STEP 8: Random Forest Model
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

print('Random Forest Results')
print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print('ROC AUC:', roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Results
Accuracy: 0.49
ROC AUC: 0.512610340479193
              precision    recall  f1-score   support

           0       0.40      0.59      0.47        39
           1       0.62      0.43      0.50        61

    accuracy                           0.49       100
   macro avg       0.51      0.51      0.49       100
weighted avg       0.53      0.49      0.49       100



In [9]:
# STEP 9: Save Best Model
joblib.dump(rf_model, 'coupon_model.pkl')
print('Model saved as coupon_model.pkl')

Model saved as coupon_model.pkl


In [12]:
streamlit_code = r'''
import streamlit as st
import pandas as pd
import joblib

# Load model
model = joblib.load("coupon_model.pkl")

st.set_page_config(page_title="Coupon Purchase Prediction", layout="centered")
st.title("Coupon Purchase Prediction App")
st.write("Predict whether a customer will use a coupon.")

# User Inputs
age = st.slider("Age", 18, 70, 30)
income = st.number_input("Income", min_value=10000, max_value=200000, value=50000)
gender = st.selectbox("Gender", ["Male", "Female"])
coupon_type = st.selectbox("Coupon Type", ["Food", "Travel", "Electronics"])
offer_duration = st.slider("Offer Duration (days)", 1, 30, 7)
store_distance = st.slider("Store Distance (km)", 1, 50, 5)
past_coupon_use = st.slider("Past Coupon Use Count", 0, 20, 3)

input_df = pd.DataFrame({
    'age':[age],
    'income':[income],
    'gender':[gender],
    'coupon_type':[coupon_type],
    'offer_duration':[offer_duration],
    'store_distance':[store_distance],
    'past_coupon_use':[past_coupon_use]
})

if st.button("Predict"):
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    if prediction == 1:
        st.success(f"Customer is likely to use the coupon. Probability: {probability:.2f}")
    else:
        st.error(f"Customer is unlikely to use the coupon. Probability: {probability:.2f}")

st.markdown("---")
st.write("Run using: streamlit run app.py")
'''
print(app.py)


NameError: name 'app' is not defined